# AI Arena — Finalize PPO Training (zip → ONNX + sanity)

Use this notebook AFTER `train_ppo.ipynb` to:

1. Load a trained `.zip` (PPO model)
2. Export it to `model.onnx` for the JS game
3. Run a 20-match sanity check vs GA-best
4. (Optional) Batch-convert all checkpoints into difficulty tiers
   (`model_easy.onnx`, `model_medium.onnx`, `model_hard.onnx`)

Self-contained — no extra files needed except the trained `.zip`.

## How to point it at your trained model

**Option A — same notebook session**: if you ran `train_ppo.ipynb` in
this same Kaggle notebook, the zips are already at `/kaggle/working/ppo_ckpt/`.
Nothing to do.

**Option B — different notebook**: in the right panel, click **+ Add Data
→ Notebook Output**, pick your training notebook, and the zips will
appear under `/kaggle/input/<your-notebook-slug>/`.

**Option C — upload zip yourself**: download `.zip` files locally, then
**+ Add Data → Upload Dataset** to push them to a Kaggle dataset, mounted
at `/kaggle/input/<dataset-name>/`.

The notebook auto-detects all three locations.

## 1. Config

In [ ]:
# === Which checkpoint to use as the MAIN model.onnx ===
# Leave as 'auto' to use ppo_combat_final.zip if found,
# otherwise the highest-numbered ppo_combat_<N>.zip
MAIN_MODEL = 'auto'

# === Sanity check ===
SANITY_MATCHES = 20

# === Output dir (where model.onnx + difficulty tiers go) ===
import os
OUTPUT_DIR = '/kaggle/working/onnx_out'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Difficulty tiers (only used if BATCH_EXPORT = True) ===
BATCH_EXPORT = True
# Mapping: tier name -> "approximate fraction through training to pick"
# We rank checkpoints by step count, then split into tiers.
TIER_FRACTIONS = {
    'easy':   0.20,    # ~20% through training
    'medium': 0.55,    # ~55% through training
    'hard':   1.00,    # final checkpoint
}

print(f'OUTPUT_DIR = {OUTPUT_DIR}')
print(f'MAIN_MODEL = {MAIN_MODEL}')
print(f'BATCH_EXPORT = {BATCH_EXPORT}, tiers = {list(TIER_FRACTIONS.keys())}')

## 2. Install dependencies

In [ ]:
%pip install -q stable-baselines3 onnx onnxruntime

## 3. Materialize `combat_env.py` (for sanity test)

In [ ]:
import os, sys, base64

COMBAT_ENV_B64 = '''...'''

_module_dir = '/kaggle/working/ai_arena_module'
os.makedirs(_module_dir, exist_ok=True)
_env_path = os.path.join(_module_dir, 'combat_env.py')
with open(_env_path, 'wb') as f:
    f.write(base64.b64decode(COMBAT_ENV_B64))
print(f'Wrote {_env_path} ({os.path.getsize(_env_path):,} bytes)')

if _module_dir not in sys.path:
    sys.path.insert(0, _module_dir)

for mod in list(sys.modules.keys()):
    if mod.startswith('combat_env'):
        del sys.modules[mod]

import combat_env
from combat_env import (
    CombatEnv, Curriculum, curriculum_for_step,
    OBS_DIM, ACTION_DIM, SQUAD_SIZE,
    make_ga_opponent,
)
print(f'combat_env loaded. OBS_DIM={OBS_DIM}, ACTION_DIM={ACTION_DIM}')

## 4. Find all checkpoints

Searches `/kaggle/working/ppo_ckpt/` and `/kaggle/input/**/` for any
`ppo_combat_*.zip` or `ppo_combat_final.zip`.

In [ ]:
import re, glob, shutil

def find_all_checkpoints():
    """Find SB3 checkpoints. Handles BOTH:
      (a) raw .zip files (e.g. /kaggle/working/ppo_ckpt/ppo_combat_*.zip)
      (b) auto-extracted directories (Kaggle expands single-zip dataset uploads
          into a directory containing policy.pth + data + ...). For these we
          re-zip into /kaggle/working/repacked_ckpts/ so PPO.load() works."""
    candidates = []
    found = set()
    search_roots = ['/kaggle/working/ppo_ckpt', '/kaggle/working', '/kaggle/input']

    # Phase 1: raw .zip files
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for path in glob.glob(os.path.join(root, '**/*.zip'), recursive=True):
            name = os.path.basename(path)
            if path in found or name.startswith('self_snap_'):
                continue
            m = re.match(r'ppo_combat_(\d+)\.zip$', name)
            if m:
                candidates.append((int(m.group(1)), path)); found.add(path)
            elif name == 'ppo_combat_final.zip':
                candidates.append((10**12, path)); found.add(path)

    # Phase 2: auto-extracted SB3 model directories
    repacked = '/kaggle/working/repacked_ckpts'
    os.makedirs(repacked, exist_ok=True)
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            # SB3 model marker: both 'policy.pth' and 'data' present
            if 'policy.pth' in filenames and 'data' in filenames:
                base = os.path.basename(dirpath)
                m = (re.search(r'ppo[_-]combat[_-]?(\d+)', base)
                     or re.search(r'(\d+)', base))
                if not m:
                    print(f'  Skip (cannot parse step #): {dirpath}')
                    continue
                step = int(m.group(1))
                zip_path = os.path.join(repacked, f'ppo_combat_{step}.zip')
                if not os.path.exists(zip_path):
                    print(f'  Re-zipping step {step:,}  {dirpath}')
                    shutil.make_archive(zip_path[:-4], 'zip', dirpath)
                if zip_path not in found:
                    candidates.append((step, zip_path)); found.add(zip_path)

    candidates.sort(key=lambda x: x[0])
    return candidates

ckpts = find_all_checkpoints()
print(f'\nFound {len(ckpts)} checkpoints:')
for step, path in ckpts:
    label = '(FINAL)' if step == 10**12 else f'{step:>12,} steps'
    print(f'  {label}  {path}  ({os.path.getsize(path)/1024:.1f} KB)')

if not ckpts:
    print()
    print('!!! NO CHECKPOINTS FOUND !!!')
    print('Diagnostic of /kaggle/input/:')
    if os.path.exists('/kaggle/input'):
        for d in sorted(os.listdir('/kaggle/input')):
            full = f'/kaggle/input/{d}'
            print(f'  {full}/')
            try:
                for item in sorted(os.listdir(full))[:6]:
                    sub = f'{full}/{item}'
                    if os.path.isdir(sub):
                        print(f'    {item}/  (dir, contents: {sorted(os.listdir(sub))[:5]})')
                    else:
                        print(f'    {item}  ({os.path.getsize(sub)/1024:.1f} KB)')
            except OSError as e:
                print(f'    (error listing: {e})')
    print()
    print('Add training output via "+ Add Data -> Notebook Output" or upload as Dataset.')
else:
    if MAIN_MODEL == 'auto':
        main_step, main_path = ckpts[-1]
    else:
        main_path = MAIN_MODEL
        main_step = -1
    print(f'\nMain model -> {main_path}')

## 5. Load main model + export ONNX

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from stable_baselines3 import PPO

print(f'Loading {main_path} ...')
model = PPO.load(main_path, device='cpu')
print(f'  Loaded. obs_space={model.observation_space}, action_space={model.action_space}')
print(f'  Policy: {type(model.policy).__name__}')

n_params = sum(p.numel() for p in model.policy.parameters())
print(f'  Parameters: {n_params:,}')


class OnnxablePolicy(nn.Module):
    def __init__(self, sb3_policy):
        super().__init__()
        self.extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net
    def forward(self, obs):
        latent_pi = self.extractor.policy_net(obs)
        logits = self.action_net(latent_pi)
        return torch.softmax(logits, dim=-1)


def export_onnx(sb3_model, out_path):
    sb3_model.policy.to('cpu')
    onnxable = OnnxablePolicy(sb3_model.policy).eval()
    dummy = torch.zeros(1, OBS_DIM, dtype=torch.float32)
    torch.onnx.export(
        onnxable, dummy, out_path,
        input_names=['obs'], output_names=['action_probs'],
        dynamic_axes={'obs': {0: 'batch'}, 'action_probs': {0: 'batch'}},
        opset_version=17,
        dynamo=False,
    )
    return os.path.getsize(out_path)


main_onnx = os.path.join(OUTPUT_DIR, 'model.onnx')
size = export_onnx(model, main_onnx)
print(f'\nMain ONNX -> {main_onnx} ({size/1024:.1f} KB)')

# Verify the ONNX runs
import onnxruntime as ort
sess = ort.InferenceSession(main_onnx, providers=['CPUExecutionProvider'])
test_in = np.random.randn(1, OBS_DIM).astype(np.float32)
out = sess.run(None, {'obs': test_in})
print(f'  inference: {test_in.shape} -> {out[0].shape}, sum={out[0][0].sum():.3f}')

## 6. (Optional) Batch-convert all checkpoints into difficulty tiers

If you have multiple `ppo_combat_<N>.zip` checkpoints, this cell ranks
them by step count and exports the ones at the configured fractions
(default: 20% / 55% / 100%) as `model_easy.onnx` / `model_medium.onnx` /
`model_hard.onnx`.

Skip this cell if you only want the main `model.onnx`.

In [ ]:
# Self-contained: re-define helpers if cell 9 didn't run
import torch
import torch.nn as nn
from stable_baselines3 import PPO

if 'OnnxablePolicy' not in dir():
    class OnnxablePolicy(nn.Module):
        def __init__(self, sb3_policy):
            super().__init__()
            self.extractor = sb3_policy.mlp_extractor
            self.action_net = sb3_policy.action_net
        def forward(self, obs):
            latent_pi = self.extractor.policy_net(obs)
            logits = self.action_net(latent_pi)
            return torch.softmax(logits, dim=-1)

if 'export_onnx' not in dir():
    def export_onnx(sb3_model, out_path):
        sb3_model.policy.to('cpu')
        onnxable = OnnxablePolicy(sb3_model.policy).eval()
        dummy = torch.zeros(1, OBS_DIM, dtype=torch.float32)
        torch.onnx.export(
            onnxable, dummy, out_path,
            input_names=['obs'], output_names=['action_probs'],
            dynamic_axes={'obs': {0: 'batch'}, 'action_probs': {0: 'batch'}},
            opset_version=17,
            dynamo=False,
        )
        return os.path.getsize(out_path)

if not BATCH_EXPORT or len(ckpts) < 2:
    print(f'BATCH_EXPORT=False or only {len(ckpts)} checkpoint(s) — skipping tier export.')
else:
    n = len(ckpts)
    print(f'Building difficulty tiers from {n} checkpoints...')
    for tier_name, frac in TIER_FRACTIONS.items():
        idx = max(0, min(n - 1, int(round(frac * (n - 1)))))
        step, path = ckpts[idx]
        label = '(FINAL)' if step == 10**12 else f'{step:,} steps'
        out = os.path.join(OUTPUT_DIR, f'model_{tier_name}.onnx')
        m = PPO.load(path, device='cpu')
        size = export_onnx(m, out)
        print(f'  {tier_name:>6} (frac {frac:.2f}, idx {idx}, {label}): {out} ({size/1024:.1f} KB)')

    print('\nAll outputs:')
    for f in sorted(os.listdir(OUTPUT_DIR)):
        full = os.path.join(OUTPUT_DIR, f)
        print(f'  {full}  ({os.path.getsize(full)/1024:.1f} KB)')

## 7. Sanity check: NN vs GA-best at deployment scale

In [ ]:
# Self-contained: load main model here if it isn't already in scope
import torch
import numpy as np
from stable_baselines3 import PPO

if 'model' not in dir() or model is None:
    print(f'Loading model from {main_path} ...')
    model = PPO.load(main_path, device='cpu')

DEFAULT_GA_GENOME = {
    'view_arc': 2.4, 'view_range': 720, 'snap_to_threat': 0.27,
    'patrol_scan_speed': 0.04, 'engage_distance': 280, 'spread': 0.07,
    'fire_cd_frames': 12, 'cover_dmg_threshold': 28, 'peek_duration': 55,
    'hide_duration': 40, 'flee_hp_pct': 0.34, 'flank_chance': 0.6,
}
ga_policy = make_ga_opponent(DEFAULT_GA_GENOME)
deploy_curriculum = curriculum_for_step(64_000_000, 64_000_000)  # final stage

print(f'Running {SANITY_MATCHES} matches vs GA-best at deployment scale...')

wins = losses = draws = 0
ep_rewards = []
kill_diffs = []

for match in range(SANITY_MATCHES):
    env = CombatEnv(opponent_policy=ga_policy,
                    curriculum=deploy_curriculum,
                    seed=20000 + match)
    obs_list = env.reset(seed=20000 + match)
    total_reward = 0.0
    for _ in range(env.match_ticks):
        actions = []
        for k in range(SQUAD_SIZE):
            obs = obs_list[k].astype(np.float32)
            action, _ = model.predict(obs, deterministic=True)
            actions.append(int(action))
        obs_list, rewards, done, info = env.step(actions)
        total_reward += sum(rewards) / SQUAD_SIZE
        if done: break
    ep_rewards.append(total_reward)
    kill_diffs.append(env.team_kills[0] - env.team_kills[1])
    if   env.team_kills[0] > env.team_kills[1]: wins   += 1
    elif env.team_kills[1] > env.team_kills[0]: losses += 1
    else:                                       draws  += 1
    if (match+1) % 5 == 0:
        print(f'  match {match+1:>2}/{SANITY_MATCHES}: kills NN={env.team_kills[0]} GA={env.team_kills[1]}'
              f'  W/L/D = {wins}/{losses}/{draws}')

print(f'\nFINAL:  {wins} W / {losses} L / {draws} D    win-rate {wins/SANITY_MATCHES*100:.0f}%')
print(f'Mean episode reward: {np.mean(ep_rewards):+.1f}')
print(f'Mean kill differential (NN - GA): {np.mean(kill_diffs):+.2f}')

if wins >= int(SANITY_MATCHES*0.7): print('[EXCELLENT] NN clearly beats GA-best.')
elif wins >= int(SANITY_MATCHES*0.5): print('[OK] NN comparable or better than GA-best.')
elif wins >= int(SANITY_MATCHES*0.3): print('[WEAK] Mixed — consider continuing training.')
else: print('[BAD] Try longer training or more BC pairs.')

## 8. Output files

In Kaggle's right-side **Output** panel, browse to `/kaggle/working/onnx_out/`:

- **`model.onnx`** ← main model. Drop into the AshGrid repo at
  `ai_arena/onnx/model.onnx` and reload the page.
- **`model_easy.onnx` / `model_medium.onnx` / `model_hard.onnx`** ← difficulty
  tiers. (Future: NN lobby will let players pick which to load — for now
  just rename whichever feels right to `model.onnx`.)

Right-click → Download from the Output panel.